### 실습 문제 1: 베이즈 정리

베이즈 정리는 분류 문제에서 특정 클래스(가설, C)가 주어진 특징(데이터/증거, F)에 대해 참일 확률을 계산하는 데 사용됩니다. 스팸 필터 예제를 들어, 이메일이 스팸(C=스팸)인지 아닌지(C=비스팸)를 특징 F(예: 특정 단어 포함 여부)에 기반해 판단한다고 합시다. 이메일이 스팸일 확률 P(C|F)는 다음과 같이 계산됩니다: :

$$ P(C|F) = \frac{P(F|C) \cdot P(C)}{P(F)} $$

여기서 각 항목의 의미는 다음과 같습니다:

- P(C|F): 주어진 특징 F(예: "무료"라는 단어 포함)를 가진 이메일이 스팸(C=스팸)일 확률 (사후 확률)
-  P(F|C): 스팸 이메일(C=스팸)에서 특징 F(예: "무료" 단어)가 나타날 확률 (우도)
-  P(C): 이메일이 스팸일 사전 확률 (전체 이메일 중 스팸 비율)
-  P(F): 특징 F가 나타날 전체 확률 (전체 이메일 중 "무료" 단어가 나타날 확률)


다음과 같은 이메일 데이터가 있다고 가정합니다.

| 이메일 내용 (키워드) | 분류   |
| :------------------- | :----- |
| '광고', '무료'       | 스팸   |
| '회의', '업무'       | 정상   |
| '광고', '당첨'       | 스팸   |
| '안부', '친구'       | 정상   |
| '출장', '제출'       | 정상   |
| '광고', '세일'       | 정상   |

이메일에 '광고'라는 단어가 포함(F)되었을 때, 그 이메일이 스팸(C=스팸)인지를 베이즈 정리를 이용하여 판단하세요.

* keywords 컬럼은 list 타입 데이터입니다. df['keywords'].apply(lambda x: '광고' in x) 로 특정 키워드가 존재하는지 여부를 확인할 수 있습니다.

In [ ]:
import pandas as pd

# Step 1. 데이터프레임 만들기
data = {
    'keywords': [['광고', '무료'], ['회의', '업무'], ['광고', '당첨'], ['안부', '친구'], ['출장', '제출'], ['광고', '세일']],
    'label': ['스팸', '정상', '스팸', '정상', '정상', '정상']
}

df = pd.DataFrame(data)
df

,keywords,label
0,"[광고, 무료]",스팸
1,"[회의, 업무]",정상
2,"[광고, 당첨]",스팸
3,"[안부, 친구]",정상
4,"[출장, 제출]",정상
5,"[광고, 세일]",정상


In [ ]:
# 라벨을 기준으로 정상과 스팸의 비율 확인
df['label'].value_counts(normalize=True)

label
정상    0.666667
스팸    0.333333
Name: proportion, dtype: float64

In [ ]:
# Step 2. '광고' 포함 여부 컬럼 추가
df['has_ad'] = df['keywords'].apply(lambda x: '광고' in x)

In [ ]:
# 1. P(스팸) — 전체에서 스팸 비율 (.mean() => 평균 내는 함수)
p_spam = (df['label'] == '스팸').mean()

# 2. P(광고) — 전체에서 광고 단어 포함된 비율
p_ad = df['has_ad'].mean()

# 3. P(광고 | 스팸) — 스팸 중 광고 포함된 비율
spam_df = df[df['label'] == '스팸']
p_ad_given_spam = spam_df['has_ad'].mean()

# Step 4. 베이즈 정리 적용
p_spam_given_ad = (p_ad_given_spam * p_spam) / p_ad

In [9]:
print(f"광고라는 단어가 포함되었을 때 스팸일 확률: {p_spam_given_ad:.2f}")

광고라는 단어가 포함되었을 때 스팸일 확률: 0.67


### 실습 문제 2: 질병에 걸렸을 확률

어떤 질병의 유병률은 1%입니다. 즉, 전체 인구 중 1%가 이 질병에 걸려 있습니다. 이 질병을 진단하기 위한 검사가 있으며, 검사의 성능은 다음과 같습니다:

$$ P(\text{질병}) = 0.01 $$

* **민감도 (Sensitivity)**: 질병이 있는 사람이 양성으로 판정될 확률은 99%입니다.

    $$P(\text{양성} | \text{질병}) = 0.99 $$

* **특이도 (Specificity)**: 질병이 없는 사람이 음성으로 판정될 확률은 98%입니다.

    $$P(\text{음성} | \text{질병 없음}) = 0.98$$

* 참고: P(양성) 은 검사에서 양성으로 나올 전체 확률로, 다음 두 가지 경우를 합한 값입니다:

    * 진짜 양성: 질병이 있고 양성으로 판정되는 경우

$$ P(질병이 있는 사람 \cap 양성) = P(질병이 있는 사람) \cdot P(양성|질병이 있는 사람) $$

    * 위양성: 질병이 없고 양성으로 판정되는 경우

$$ P(질병이 없는 사람 \cap 양성) = P(질병이 없는 사람) \cdot P(양성|질병이 없는 사람) $$

어떤 사람이 이 검사를 받고 양성 판정을 받았습니다. 이 사람이 실제로 질병에 걸렸을 확률 $P(\text{질병} | \text{양성})$을 베이즈 정리를 사용하여 계산하세요.

### 실습 문제 1: 아이리스 데이터셋과 나이브 베이즈 분류기

#### 1. 데이터 준비 및 탐색

필요한 라이브러리를 불러오고 Iris 데이터를 준비합니다. (sklearn.datasets.load_iris)

* 특성과 레이블을 구분하여 각각 DataFrame 과 Series로 만듭니다. (X, y)
* train_test_split 함수를 사용하여 80%는 학습용, 20%는 테스트용으로 분리합니다.


#### 2. 나이브 베이즈 직접 계산해보기
나이브 베이즈는 베이즈 정리에 기반합니다:
$$P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}$$


* $P(C)$: 사전 확률 (Prior - 데이터와 관계없이 클래스 C일 확률)
* $P(X|C)$: 가능도 (Likelihood - 클래스 C일 때 데이터 X가 관찰될 likelihood)
* $P(C|X)$: 사후 확률 (데이터 X가 주어졌을 때 클래스 C일 확률) - 우리가 예측하려는 값!

* 참고: $P(X)$: 데이터 X가 관찰될 likelihood (보통 계산을 생략함, C 와는 상관없이 고정값)
$$ P(X) = \sum_{k} P(X|C_k) \cdot P(C_k) $$

#### 2.1 각 클래스별 사전 확률 $P(C)$ 계산하기

#### 2.2 각 클래스 및 특성별 평균($\mu$)과 표준편차($\sigma$) 계산하기

#### 2.3 테스트 데이터 샘플 하나에 대한 가능도 계산해보기

$$ P(X|C) = P(x_1|C)
\times P(x_2|C) \times P(x_3|C) \times P(x_4|C) $$

* 참고: 가우시안 분포의 PDF(확률 밀도 함수)를 계산하기 위해 scipy.stats.norm.pdf 함수를 사용해 보세요.


#### 2.4 테스트 데이터 샘플 하나에 대한 사후 확률($P(C|X)$) 계산해보기